In [32]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ============================================================
# 1. SETTINGS
# ============================================================

kernel = "rbf"
nu = 0.05
gamma = "auto"

threshold_list = [0.0, 10.0]

train_cycle_fraction = 0.80

total_normal_samples = 50000
total_anomaly_samples = 10000

train_normal_samples = int(total_normal_samples * train_cycle_fraction)
test_normal_samples = total_normal_samples - train_normal_samples
test_anomaly_samples = total_anomaly_samples

n_runs = 5
seeds = [42, 43, 44, 45, 46]

data_path = Path("../data_processed/NASA/nasa_battery_features_rolling.csv")
results_root = Path("../results/OCSVM_NASA_random_files_5runs")
results_root.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. LOAD DATA
# ============================================================

df = pd.read_csv(data_path)

feature_cols = [
    "V", "I", "T",
    "Current_load", "Voltage_load", "Time",
    "V_diff1", "I_diff1", "T_diff1",
    "V_roll_mean", "V_roll_std", "V_roll_min", "V_roll_max",
    "I_roll_mean", "I_roll_std",
    "T_roll_mean", "T_roll_std"
]

required_cols = ["source_file", "y_true_file"] + feature_cols
df = df.dropna(subset=required_cols).reset_index(drop=True)

df["source_file_num"] = (
    df["source_file"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

df = df.sort_values(["source_file_num", "Time"]).reset_index(drop=True)

file_info = (
    df.groupby("source_file", as_index=False)
      .agg(
          source_file_num=("source_file_num", "first"),
          y_true_file=("y_true_file", "first"),
          n_rows=("source_file", "size")
      )
      .sort_values("source_file_num")
      .reset_index(drop=True)
)

normal_files_info = file_info[file_info["y_true_file"] == 0].reset_index(drop=True)
anomaly_files_info = file_info[file_info["y_true_file"] == 1].reset_index(drop=True)

print("Normal files total:", len(normal_files_info))
print("Anomaly files total:", len(anomaly_files_info))
print("Train normal samples:", train_normal_samples)
print("Test normal samples:", test_normal_samples)
print("Test anomaly samples:", test_anomaly_samples)

# ============================================================
# 3. HELPER
# ============================================================

def safe_name(value):
    return str(value).replace(".", "_").replace("-", "minus_")

all_results = []

# ============================================================
# 4. 5 RANDOM RUNS BY SOURCE_FILE
# ============================================================

for run_id, seed in enumerate(seeds, start=1):
    print("\n================================================")
    print(f"RUN {run_id}/{n_runs}, seed = {seed}")
    print("================================================")

    run_dir = results_root / f"run_{run_id}_seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    normal_files_shuffled = normal_files_info.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    split_normal_idx = int(len(normal_files_shuffled) * train_cycle_fraction)

    train_normal_files = normal_files_shuffled.iloc[:split_normal_idx]["source_file"]
    test_normal_files = normal_files_shuffled.iloc[split_normal_idx:]["source_file"]

    anomaly_files_shuffled = anomaly_files_info.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    test_anomaly_files = anomaly_files_shuffled["source_file"]

    train_window = df[df["source_file"].isin(train_normal_files)].copy()
    test_normal_window = df[df["source_file"].isin(test_normal_files)].copy()
    test_anomaly_window = df[df["source_file"].isin(test_anomaly_files)].copy()

    train_normal = (
        train_window[train_window["y_true_file"] == 0]
        .iloc[:train_normal_samples]
        .copy()
    )

    test_normal = (
        test_normal_window[test_normal_window["y_true_file"] == 0]
        .iloc[:test_normal_samples]
        .copy()
    )

    test_anomaly = (
        test_anomaly_window[test_anomaly_window["y_true_file"] == 1]
        .iloc[:test_anomaly_samples]
        .copy()
    )

    test_data = pd.concat(
        [test_normal, test_anomaly],
        ignore_index=True
    )

    X_train_normal_raw = train_normal[feature_cols].copy()
    X_test_raw = test_data[feature_cols].copy()
    y_test = test_data["y_true_file"].astype(int).copy()

    print("Split type: random split by complete source_file cycles")
    print("Train normal files:", train_normal["source_file"].nunique())
    print("Test normal files:", test_normal["source_file"].nunique())
    print("Test anomaly files:", test_anomaly["source_file"].nunique())
    print("Train normal samples:", len(train_normal))
    print("Test normal samples:", len(test_normal))
    print("Test anomaly samples:", len(test_anomaly))
    print("Test total samples:", len(test_data))

    # ========================================================
    # Scaling
    # ========================================================

    scaler = StandardScaler()

    X_train = pd.DataFrame(
        scaler.fit_transform(X_train_normal_raw),
        columns=feature_cols,
        index=X_train_normal_raw.index
    )

    X_test = pd.DataFrame(
        scaler.transform(X_test_raw),
        columns=feature_cols,
        index=X_test_raw.index
    )

    # ========================================================
    # Train model
    # ========================================================

    model = OneClassSVM(
        kernel=kernel,
        nu=nu,
        gamma=gamma
    )

    model.fit(X_train)

    decision_scores = model.decision_function(X_test).ravel()
    score_samples = model.score_samples(X_test).ravel()
    pred_default = model.predict(X_test)

    print("Decision score min:", decision_scores.min())
    print("Decision score max:", decision_scores.max())
    print("Decision score mean:", decision_scores.mean())

    # ========================================================
    # Threshold experiments
    # ========================================================

    for threshold in threshold_list:
        exp_name = (
            f"run_{run_id}_kernel_{kernel}_nu_{safe_name(nu)}"
            f"_gamma_{gamma}_threshold_{safe_name(threshold)}"
        )

        exp_dir = run_dir / exp_name
        exp_dir.mkdir(parents=True, exist_ok=True)

        y_pred = (decision_scores < threshold).astype(int)

        df_results = test_data.copy()
        df_results["ocsvm_pred_default"] = pred_default
        df_results["ocsvm_decision"] = decision_scores
        df_results["ocsvm_score"] = score_samples
        df_results["threshold"] = threshold
        df_results["is_anomaly"] = y_pred

        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        accuracy = accuracy_score(y_test, y_pred) * 100
        precision = precision_score(y_test, y_pred, zero_division=0) * 100
        recall = recall_score(y_test, y_pred, zero_division=0) * 100
        f1 = f1_score(y_test, y_pred, zero_division=0) * 100

        result_row = {
            "run_id": run_id,
            "seed": seed,
            "experiment": exp_name,
            "kernel": kernel,
            "nu": nu,
            "gamma": gamma,
            "threshold": threshold,
            "split_type": "random_source_file_split_5runs",
            "train_cycle_fraction": train_cycle_fraction,
            "total_normal_samples": total_normal_samples,
            "total_anomaly_samples": total_anomaly_samples,
            "train_normal_samples": len(train_normal),
            "test_normal_samples": len(test_normal),
            "test_anomaly_samples": len(test_anomaly),
            "test_total_samples": len(test_data),
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
            "accuracy_%": round(accuracy, 2),
            "precision_%": round(precision, 2),
            "recall_%": round(recall, 2),
            "f1_%": round(f1, 2)
        }

        all_results.append(result_row)

        df_results.to_csv(exp_dir / "predictions.csv", index=False)

        print("======================================")
        print("One-Class SVM NASA Battery")
        print("======================================")
        print(f"Run      : {run_id}")
        print(f"Seed     : {seed}")
        print(f"Threshold: {threshold}")
        print(f"Accuracy : {accuracy:.2f}%")
        print(f"Precision: {precision:.2f}%")
        print(f"Recall   : {recall:.2f}%")
        print(f"F1-score : {f1:.2f}%")
        print("Confusion Matrix")
        print(cm)

        # Metrics table
        metrics_df = pd.DataFrame({
            "Metrika": [
                "Presnosť (Accuracy)",
                "Precíznosť (Precision)",
                "Citlivosť (Recall)",
                "F1-skóre"
            ],
            "Hodnota (%)": [
                round(accuracy, 2),
                round(precision, 2),
                round(recall, 2),
                round(f1, 2)
            ]
        })

        fig, ax = plt.subplots(figsize=(6, 2))
        ax.axis("off")

        table = ax.table(
            cellText=metrics_df.values,
            colLabels=metrics_df.columns,
            loc="center",
            cellLoc="center"
        )

        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 2)

        plt.title(
            f"Výsledné metriky One-Class SVM – NASA\n"
            f"Run = {run_id}, Threshold = {threshold}",
            fontsize=12,
            pad=20
        )

        plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
        plt.close()

        # Confusion matrix
        plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title(f"Matica zámen – Run {run_id}, Threshold = {threshold}")
        plt.colorbar()

        classes = ["Normálne", "Anomálie"]
        tick_marks = np.arange(len(classes))

        plt.xticks(tick_marks, classes)
        plt.yticks(tick_marks, classes)

        threshold_cm = cm.max() / 2.0 if cm.max() > 0 else 0.5

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                color = "black" if cm[i, j] > threshold_cm else "white"
                plt.text(j, i, cm[i, j], ha="center", va="center", color=color)

        plt.xlabel("Predikované triedy")
        plt.ylabel("Skutočné triedy")
        plt.tight_layout()
        plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close()

        # Score histogram
        plt.figure(figsize=(10, 5))

        plt.hist(
            decision_scores[y_test.values == 0],
            bins=50,
            alpha=0.7,
            label="Normálne"
        )

        plt.hist(
            decision_scores[y_test.values == 1],
            bins=50,
            alpha=0.7,
            label="Anomálie"
        )

        plt.axvline(
            threshold,
            linestyle="--",
            linewidth=2,
            label=f"Threshold = {threshold}"
        )

        plt.title(
            f"Histogram skóre One-Class SVM – NASA\n"
            f"Run = {run_id}, Threshold = {threshold}"
        )
        plt.xlabel("Decision score")
        plt.ylabel("Počet vzoriek")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(exp_dir / "score_histogram.png", dpi=300, bbox_inches="tight")
        plt.close()

# ====================================================
# LOWEST SCORE ANALYSIS
# ====================================================

score_df = test_data.copy()
score_df["decision_score"] = decision_scores
score_df["score_samples"] = score_samples
score_df["threshold"] = threshold
score_df["predicted_anomaly"] = y_pred

score_df = score_df.sort_values(
    by="decision_score",
    ascending=True
).reset_index(drop=True)

cols_to_show = [
    "source_file",
    "y_true_file",
    "decision_score",
    "score_samples",
    "V",
    "I",
    "T",
    "Voltage_load",
    "Current_load",
    "Time"
]

print("\n================================")
print(f"LOWEST SCORES ANALYSIS | Run {run_id} | Threshold {threshold}")
print("================================")

print("\n20 LOWEST SCORES:")
print(score_df[cols_to_show].head(20))

score_df[cols_to_show].head(200).to_csv(
    exp_dir / "lowest_scores.csv",
    index=False
)

lowest_files_counts = (
    score_df.head(500)["source_file"]
    .value_counts()
    .reset_index()
)

lowest_files_counts.columns = ["source_file", "count_in_lowest_500"]

lowest_files_counts.to_csv(
    exp_dir / "lowest_score_files.csv",
    index=False
)

print("\nFILES IN LOWEST 500 SCORES:")
print(lowest_files_counts.head(20))

# ====================================================
# HIGHEST SCORES ANALYSIS
# ====================================================

high_scores = score_df.sort_values(
    by="decision_score",
    ascending=False
).reset_index(drop=True)

cols_to_show = [
    "source_file",
    "y_true_file",
    "decision_score",
    "score_samples",
    "V",
    "I",
    "T",
    "Voltage_load",
    "Current_load",
    "Time"
]

print("\n================================")
print(f"HIGHEST SCORES ANALYSIS | Run {run_id} | Threshold {threshold}")
print("================================")

print("\n20 HIGHEST SCORES:")
print(
    high_scores[cols_to_show]
    .head(20)
)

high_scores[cols_to_show].head(200).to_csv(
    exp_dir / "highest_scores.csv",
    index=False
)

# ====================================================
# ONLY ANOMALIES WITH VERY HIGH SCORES
# ====================================================

high_score_anomalies = high_scores[
    high_scores["y_true_file"] == 1
].copy()

print("\n================================")
print("TOP ANOMALIES WITH HIGHEST SCORES")
print("================================")

print(
    high_score_anomalies[cols_to_show]
    .head(50)
)

high_score_anomalies[cols_to_show].head(500).to_csv(
    exp_dir / "highest_score_anomalies.csv",
    index=False
)

# ====================================================
# WHICH FILES CREATE THESE ANOMALIES?
# ====================================================

anomaly_file_counts = (
    high_score_anomalies.head(500)["source_file"]
    .value_counts()
    .reset_index()
)

anomaly_file_counts.columns = [
    "source_file",
    "count_in_top_500_high_scores"
]

print("\n================================")
print("FILES OF NORMAL-LIKE ANOMALIES")
print("================================")

print(anomaly_file_counts.head(20))

anomaly_file_counts.to_csv(
    exp_dir / "highest_score_anomaly_files.csv",
    index=False
)

# ====================================================
# BOXPLOT OF MOST NORMAL-LIKE ANOMALY FILES
# ====================================================

top_files = (
    anomaly_file_counts
    .head(10)["source_file"]
    .tolist()
)

if len(top_files) > 0:

    plt.figure(figsize=(12, 6))

    high_score_anomalies[
        high_score_anomalies["source_file"].isin(top_files)
    ].boxplot(
        column="decision_score",
        by="source_file",
        rot=90
    )

    plt.title(
        "Najviac normálne vyzerajúce anomálne súbory"
    )

    plt.suptitle("")
    plt.xlabel("Source file")
    plt.ylabel("Decision score")

    plt.tight_layout()

    plt.savefig(
        exp_dir / "highest_score_anomaly_files_boxplot.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()
    
# ====================================================
# BOXPLOT OF WORST FILES
# ====================================================

worst_files = (
    score_df.head(500)["source_file"]
    .value_counts()
    .head(10)
    .index
)

plt.figure(figsize=(12, 6))

score_df[
    score_df["source_file"].isin(worst_files)
].boxplot(
    column="decision_score",
    by="source_file",
    rot=90
)

plt.title("Decision score podľa source_file – najnižšie skóre")
plt.suptitle("")
plt.xlabel("Source file")
plt.ylabel("Decision score")
plt.tight_layout()

plt.savefig(
    exp_dir / "lowest_score_files_boxplot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ====================================================
# SCATTER PLOTS: FEATURES VS SCORE
# ====================================================

for col in ["V", "I", "T", "Voltage_load", "Current_load"]:
    plt.figure(figsize=(8, 5))

    normal_mask = score_df["y_true_file"] == 0
    anomaly_mask = score_df["y_true_file"] == 1

    plt.scatter(
        score_df.loc[normal_mask, col],
        score_df.loc[normal_mask, "decision_score"],
        alpha=0.25,
        s=8,
        label="Normálne"
    )

    plt.scatter(
        score_df.loc[anomaly_mask, col],
        score_df.loc[anomaly_mask, "decision_score"],
        alpha=0.25,
        s=8,
        label="Anomálie"
    )

    plt.axhline(
        threshold,
        linestyle="--",
        linewidth=2,
        label=f"Threshold = {threshold}"
    )

    plt.xlabel(col)
    plt.ylabel("Decision score")
    plt.title(f"{col} vs Decision score – Run {run_id}, Threshold = {threshold}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    plt.savefig(
        exp_dir / f"score_vs_{col}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()
    
# ============================================================
# 5. SAVE SUMMARY AND MEAN RESULTS FOR ALL THRESHOLDS
# ============================================================

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by=["threshold", "run_id"]).reset_index(drop=True)

results_df.to_csv(
    results_root / "summary_results_all_runs_all_thresholds.csv",
    index=False
)

mean_results_df = (
    results_df
    .groupby(["kernel", "nu", "gamma", "threshold"], as_index=False)
    .agg({
        "accuracy_%": ["mean", "std"],
        "precision_%": ["mean", "std"],
        "recall_%": ["mean", "std"],
        "f1_%": ["mean", "std"],
        "tn": "mean",
        "fp": "mean",
        "fn": "mean",
        "tp": "mean"
    })
)

mean_results_df.columns = [
    "kernel", "nu", "gamma", "threshold",
    "accuracy_mean_%", "accuracy_std_%",
    "precision_mean_%", "precision_std_%",
    "recall_mean_%", "recall_std_%",
    "f1_mean_%", "f1_std_%",
    "tn_mean", "fp_mean", "fn_mean", "tp_mean"
]

mean_results_df = mean_results_df.sort_values(
    by="threshold",
    ascending=True
).reset_index(drop=True)

mean_results_df.to_csv(
    results_root / "summary_results_mean_5runs_all_thresholds.csv",
    index=False
)

# ============================================================
# FINAL RESULTS FOLDER
# ============================================================

final_dir = results_root / "FINAL_AVERAGE_RESULTS_ALL_THRESHOLDS"
final_dir.mkdir(parents=True, exist_ok=True)

mean_results_df.to_csv(
    final_dir / "mean_metrics_all_thresholds.csv",
    index=False
)

# ============================================================
# FINAL METRICS TABLE IMAGE FOR ALL THRESHOLDS
# ============================================================

final_table_df = mean_results_df[[
    "threshold",
    "accuracy_mean_%",
    "accuracy_std_%",
    "precision_mean_%",
    "precision_std_%",
    "recall_mean_%",
    "recall_std_%",
    "f1_mean_%",
    "f1_std_%"
]].copy()

final_table_df = final_table_df.round(2)

final_table_df.to_csv(
    final_dir / "final_metrics_table_all_thresholds.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(14, 3))
ax.axis("off")

table = ax.table(
    cellText=final_table_df.values,
    colLabels=final_table_df.columns,
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)

plt.title(
    "Priemerné výsledky One-Class SVM – NASA (5 behov, rôzne thresholdy)",
    fontsize=12,
    pad=20
)

plt.savefig(
    final_dir / "final_metrics_table_all_thresholds.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ============================================================
# AVERAGE CONFUSION MATRIX FOR EACH THRESHOLD
# ============================================================

for _, row in mean_results_df.iterrows():
    threshold = row["threshold"]

    threshold_dir = final_dir / f"threshold_{safe_name(threshold)}"
    threshold_dir.mkdir(parents=True, exist_ok=True)

    avg_cm = np.array([
        [row["tn_mean"], row["fp_mean"]],
        [row["fn_mean"], row["tp_mean"]]
    ])

    avg_cm_df = pd.DataFrame(
        avg_cm,
        index=["Skutočné normálne", "Skutočné anomálie"],
        columns=["Predikované normálne", "Predikované anomálie"]
    )

    avg_cm_df.to_csv(
        threshold_dir / "average_confusion_matrix.csv"
    )

    plt.figure(figsize=(6, 5))
    plt.imshow(avg_cm, interpolation="nearest")
    plt.title(f"Priemerná matica zámen – Threshold = {threshold}")
    plt.colorbar()

    classes = ["Normálne", "Anomálie"]
    tick_marks = np.arange(len(classes))

    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    threshold_cm = avg_cm.max() / 2.0 if avg_cm.max() > 0 else 0.5

    for i in range(avg_cm.shape[0]):
        for j in range(avg_cm.shape[1]):
            color = "black" if avg_cm[i, j] > threshold_cm else "white"
            plt.text(
                j,
                i,
                f"{avg_cm[i, j]:.1f}",
                ha="center",
                va="center",
                color=color
            )

    plt.xlabel("Predikované triedy")
    plt.ylabel("Skutočné triedy")
    plt.tight_layout()

    plt.savefig(
        threshold_dir / "average_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    threshold_metrics = pd.DataFrame({
        "Metrika": [
            "Accuracy mean (%)",
            "Accuracy std",
            "Precision mean (%)",
            "Precision std",
            "Recall mean (%)",
            "Recall std",
            "F1 mean (%)",
            "F1 std"
        ],
        "Hodnota": [
            round(row["accuracy_mean_%"], 2),
            round(row["accuracy_std_%"], 2),
            round(row["precision_mean_%"], 2),
            round(row["precision_std_%"], 2),
            round(row["recall_mean_%"], 2),
            round(row["recall_std_%"], 2),
            round(row["f1_mean_%"], 2),
            round(row["f1_std_%"], 2)
        ]
    })

    threshold_metrics.to_csv(
        threshold_dir / "average_metrics.csv",
        index=False
    )

    # ============================================================
    # FINAL METRICS TABLE (WITHOUT STD)
    # ============================================================
    
    final_metrics_only_mean = pd.DataFrame({
        "Metrika": [
            "Accuracy (%)",
            "Precision (%)",
            "Recall (%)",
            "F1-score (%)"
        ],
        "Hodnota": [
            round(row["accuracy_mean_%"], 2),
            round(row["precision_mean_%"], 2),
            round(row["recall_mean_%"], 2),
            round(row["f1_mean_%"], 2)
        ]
    })
    
    final_metrics_only_mean.to_csv(
        threshold_dir / "final_metrics_table.csv",
        index=False
    )
    
    fig, ax = plt.subplots(figsize=(6, 2))
    ax.axis("off")
    
    table = ax.table(
        cellText=final_metrics_only_mean.values,
        colLabels=final_metrics_only_mean.columns,
        loc="center",
        cellLoc="center"
    )
    
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)
    
    plt.title(
        f"Priemerné metriky One-Class SVM\nThreshold = {threshold}",
        fontsize=12,
        pad=20
    )
    
    plt.savefig(
        threshold_dir / "final_metrics_table.png",
        dpi=300,
        bbox_inches="tight"
    )
    
    plt.close()

print("\n==============================")
print("MEAN RESULTS OVER 5 RUNS FOR ALL THRESHOLDS")
print("==============================")
print(mean_results_df)

print("Done")

Normal files total: 2215
Anomaly files total: 517
Train normal samples: 40000
Test normal samples: 10000
Test anomaly samples: 10000

RUN 1/5, seed = 42
Split type: random split by complete source_file cycles
Train normal files: 131
Test normal files: 33
Test anomaly files: 33
Train normal samples: 40000
Test normal samples: 10000
Test anomaly samples: 10000
Test total samples: 20000
Decision score min: -194.35757719979475
Decision score max: 53.05751805485144
Decision score mean: -22.272883667865397
One-Class SVM NASA Battery
Run      : 1
Seed     : 42
Threshold: 0.0
Accuracy : 78.31%
Precision: 93.43%
Recall   : 60.89%
F1-score : 73.73%
Confusion Matrix
[[9572  428]
 [3911 6089]]
One-Class SVM NASA Battery
Run      : 1
Seed     : 42
Threshold: 10.0
Accuracy : 81.05%
Precision: 89.13%
Recall   : 70.74%
F1-score : 78.88%
Confusion Matrix
[[9137  863]
 [2926 7074]]

RUN 2/5, seed = 43
Split type: random split by complete source_file cycles
Train normal files: 130
Test normal files: 34
T

<Figure size 1200x600 with 0 Axes>

<Figure size 1200x600 with 0 Axes>